In [1]:
!pip install crewai crewai-tools litellm langchain-groq pypdf -q

This cell installs the necessary Python packages: `crewai`, `crewai-tools`, `litellm`, `langchain-groq`, and `pypdf`. The `-q` flag ensures a quiet installation without verbose output.

In [10]:
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

This cell imports and configures a caching mechanism for CrewAI's LLM interactions. It helps manage and potentially optimize repeated LLM calls internally.

In [12]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

This cell imports the `os` module and `userdata` from `google.colab`. It then sets the `GROQ_API_KEY` environment variable using a secret stored in Google Colab's user data, which is essential for authenticating with the Groq API.

In [13]:
from crewai import LLM

llm = LLM(
    model="groq/openai/gpt-oss-120b",
    temperature=0.3
)

This cell initializes a `crewai` `LLM` (Large Language Model) object. It's configured to use the `groq/openai/gpt-oss-120b` model with a `temperature` of 0.3, controlling the creativity of the LLM's responses.

In [14]:
from crewai.tools import BaseTool
from crewai import LLM

class ResumeParserTool(BaseTool):
    name: str = "Resume Parser"
    description: str = "Extracts skills, experience, and education from resume text or PDF."

    def _run(self, resume_text: str) -> str:
        parser_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.1)

        prompt = f"""
        Extract the following from this resume as clean bullet points:
        - Skills
        - Experience (roles, companies, years)
        - Education

        Resume:
        {resume_text}
        """

        response = parser_llm.call(prompt)
        return response

This cell defines a custom `ResumeParserTool` for CrewAI. This tool uses an LLM to extract key information like skills, experience, and education from a given resume text, formatting it into clean bullet points.

In [15]:
from crewai.tools import BaseTool
from crewai import LLM

class JobMatchComparatorTool(BaseTool):
    name: str = "Job Match Comparator"
    description: str = "Compares parsed resume data against a job description and returns a match score with gaps."

    def _run(self, parsed_resume: str, job_description: str) -> str:
        comparator_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.2)

        prompt = f"""
        Compare the parsed resume below against the job description.
        Return:
        1. A match score out of 100
        2. Matching skills/experience
        3. Missing skills/requirements
        4. A short recommendation for the candidate

        Parsed Resume:
        {parsed_resume}

        Job Description:
        {job_description}
        """

        response = comparator_llm.call(prompt)
        return response

This cell defines another custom CrewAI tool, `JobMatchComparatorTool`. It takes parsed resume data and a job description, then uses an LLM to compare them, generating a match score, identifying matching and missing skills, and providing a short recommendation.

In [16]:
from crewai import Agent, Task, Crew
matcher_agent = Agent(
    role="Resume Matching Specialist",
    goal="Parse resumes and accurately match them against job descriptions",
    backstory="An experienced HR tech specialist who evaluates resumes against job requirements with precision.",
    tools=[ResumeParserTool(), JobMatchComparatorTool()],
    llm=llm,
    verbose=True
)

This cell defines a `matcher_agent` for the CrewAI framework. This agent is designated as a 'Resume Matching Specialist' with the goal of parsing and matching resumes against job descriptions. It is equipped with the `ResumeParserTool` and `JobMatchComparatorTool`.

In [17]:
matching_task = Task(
    description=(
        "Given the resume text: {resume_text} and job description: {job_description}, "
        "first parse the resume using the Resume Parser tool, "
        "then compare the parsed result against the job description using the Job Match Comparator tool. "
        "Return the final match report."
    ),
    expected_output="A match score, matching points, missing points, and a recommendation.",
    agent=matcher_agent
)

This cell defines the `matching_task` that the `matcher_agent` will perform. The task involves using the defined tools to first parse a resume and then compare it against a job description, ultimately returning a detailed match report.

In [18]:
crew = Crew(
    agents=[matcher_agent],
    tasks=[matching_task],
    verbose=True
)

This cell creates a `Crew` instance, which orchestrates the agents and tasks. Here, it's set up with the `matcher_agent` and the `matching_task`.

In [19]:
sample_resume = """
Waleed - AI/ML Engineer
Skills: Python, TensorFlow, CrewAI, SQL, Power BI
Experience: 2 years as Data Analyst at CoreVizion, built ML pipelines and dashboards.
Education: BS in Computer Science
"""

sample_job = """
Looking for an Agentic AI Engineer with experience in Python, LLMs,
multi-agent frameworks (CrewAI/LangGraph), and cloud deployment.
"""

result = await crew.kickoff_async(inputs={
    "resume_text": sample_resume,
    "job_description": sample_job
})

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 18638627-a9fe-4f19-b077-cc07bed9c18c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Given the resume text:                                                                                   │
│  Waleed - AI/ML Engineer                                                                                        │
│  Skills: Python, TensorFlow, CrewAI, SQL, Power BI                                                              │
│  Experience: 2 years as Data Analyst at CoreVizion, built ML pipelines and dashboards.                          │
│  Education: BS in Computer Science                                                                              │
│   and job description:                                                                                          │
│  Looking for an Agentic AI Engineer with experience in Python, LLMs,                                            │
│  multi-agent frameworks (CrewAI/LangGraph), and cloud deployment.                                               │
│  , first parse the resume using the Resume Parser tool, then compare the parsed result against the job          │
│  description using the Job Match Comparator tool. Return the final match report.                                │
│  ID: 9d1098f1-02d7-4446-9996-e1381740079d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Matching Specialist                                                                              │
│                                                                                                                 │
│  Task: Given the resume text:                                                                                   │
│  Waleed - AI/ML Engineer                                                                                        │
│  Skills: Python, TensorFlow, CrewAI, SQL, Power BI                                                              │
│  Experience: 2 years as Data Analyst at CoreVizion, built ML pipelines and dashboards.                          │
│  Education: BS in Computer Science                                                                              │
│   and job description:                                                                                          │
│  Looking for an Agentic AI Engineer with experience in Python, LLMs,                                            │
│  multi-agent frameworks (CrewAI/LangGraph), and cloud deployment.                                               │
│  , first parse the resume using the Resume Parser tool, then compare the parsed result against the job          │
│  description using the Job Match Comparator tool. Return the final match report.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: resume_parser                                                                                            │
│  Args: {'resume_text': 'Waleed - AI/ML Engineer\nSkills: Python, TensorFlow, CrewAI, SQL, Power                 │
│  BI\nExperience: 2 years as Data Analyst at CoreVizion, built ML pipelines and dashboards.\nEducation: BS in    │
│  Com...                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool resume_parser executed with result: **Skills**
- Python  
- TensorFlow  
- CrewAI  
- SQL  
- Power BI  

**Experience**
- **Data Analyst**, CoreVizion – 2 years  
  - Built machine‑learning pipelines and dashboards  

**Education**
- B...


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Matching Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Match Score:** **65 / 100**                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Matching Skills / Experience                                                                            │
│  | Resume Item | How it Matches the JD |                                                                        │
│  |-------------|-----------------------|                                                                        │
│  | **Python** | Required core language – ✅ |                                                                   │
│  | **CrewAI** | One of the listed multi‑agent frameworks – ✅ |                                                 │
│  | **Machine‑learning pipelines (TensorFlow)** | Demonstrates ability to build AI/ML components – ✅ |          │
│  | **2 years as Data Analyst building ML pipelines & dashboards** | Relevant hands‑on experience with           │
│  data‑driven AI work – ✅ |                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Missing Skills / Requirements                                                                           │
│  | Required Item | Not Evident in Resume |                                                                      │
│  |---------------|-----------------------|                                                                      │
│  | **Large Language Models (LLMs)** | No mention of working with LLMs (e.g., GPT, BERT, etc.) |                 │
│  | **Other multi‑agent framework (LangGraph)** | Only CrewAI is listed |                                        │
│  | **Cloud deployment / DevOps** (AWS, GCP, Azure, Docker, Kubernetes, etc.) | No cloud or containerisation     │
│  experience shown |                                                                                             │
│  | **Production‑grade AI engineering practices** (CI/CD, monitoring, scaling) | Not indicated |                 │
│  | **Relevant advanced AI research or engineering projects** | Not present |                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Recommendation                                                                                          │
│  The candidate has a solid foundation in Python and practical experience with the CrewAI multi‑agent framework  │
│  and ML pipeline development, which aligns well with the co

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Given the resume text:                                                                                   │
│  Waleed - AI/ML Engineer                                                                                        │
│  Skills: Python, TensorFlow, CrewAI, SQL, Power BI                                                              │
│  Experience: 2 years as Data Analyst at CoreVizion, built ML pipelines and dashboards.                          │
│  Education: BS in Computer Science                                                                              │
│   and job description:                                                                                          │
│  Looking for an Agentic AI Engineer with experience in Python, LLMs,                                            │
│  multi-agent frameworks (CrewAI/LangGraph), and cloud deployment.                                               │
│  , first parse the resume using the Resume Parser tool, then compare the parsed result against the job          │
│  description using the Job Match Comparator tool. Return the final match report.                                │
│  Agent: Resume Matching Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 18638627-a9fe-4f19-b077-cc07bed9c18c                                                                       │
│  Final Output: **Match Score:** **65 / 100**                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Matching Skills / Experience                                                                            │
│  | Resume Item | How it Matches the JD |                                                                        │
│  |-------------|-----------------------|                                                                        │
│  | **Python** | Required core language – ✅ |                                                                   │
│  | **CrewAI** | One of the listed multi‑agent frameworks – ✅ |                                                 │
│  | **Machine‑learning pipelines (TensorFlow)** | Demonstrates ability to build AI/ML components – ✅ |          │
│  | **2 years as Data Analyst building ML pipelines & dashboards** | Relevant hands‑on experience with           │
│  data‑driven AI work – ✅ |                                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Missing Skills / Requirements                                                                           │
│  | Required Item | Not Evident in Resume |                                                                      │
│  |---------------|-----------------------|                                                                      │
│  | **Large Language Models (LLMs)** | No mention of working with LLMs (e.g., GPT, BERT, etc.) |                 │
│  | **Other multi‑agent framework (LangGraph)** | Only CrewAI is listed |                                        │
│  | **Cloud deployment / DevOps** (AWS, GCP, Azure, Docker, Kubernetes, etc.) | No cloud or containerisation     │
│  experience shown |                                                                                             │
│  | **Production‑grade AI engineering practices** (CI/CD, monitoring, scaling) | Not indicated |                 │
│  | **Relevant advanced AI research or engineering projects** | Not present |                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Recommendation                                                                                          │
│  The candidate has a solid foundation in Python and practical experience with the CrewAI multi‑agent framework  │
│  and ML pipeline development, which aligns well with the c

**Match Score:** **65 / 100**

---

### 1. Matching Skills / Experience
| Resume Item | How it Matches the JD |
|-------------|-----------------------|
| **Python** | Required core language – ✅ |
| **CrewAI** | One of the listed multi‑agent frameworks – ✅ |
| **Machine‑learning pipelines (TensorFlow)** | Demonstrates ability to build AI/ML components – ✅ |
| **2 years as Data Analyst building ML pipelines & dashboards** | Relevant hands‑on experience with data‑driven AI work – ✅ |

---

### 2. Missing Skills / Requirements
| Required Item | Not Evident in Resume |
|---------------|-----------------------|
| **Large Language Models (LLMs)** | No mention of working with LLMs (e.g., GPT, BERT, etc.) |
| **Other multi‑agent framework (LangGraph)** | Only CrewAI is listed |
| **Cloud deployment / DevOps** (AWS, GCP, Azure, Docker, Kubernetes, etc.) | No cloud or containerisation experience shown |
| **Production‑grade AI engineering practices** (CI/CD, monitoring, scaling) | Not indicated |

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

This cell defines a `sample_resume` and `sample_job` as multi-line strings. It then initiates the CrewAI workflow by calling `crew.kickoff_async` with these sample inputs and prints the resulting match report generated by the agent.

In [20]:
!pip install streamlit -q

This cell installs the Streamlit library, which is used for building interactive web applications, quietly.

In [21]:
!streamlit --version

Streamlit, version 1.61.1


This cell checks the installed version of Streamlit, confirming that it's `Streamlit, version 1.61.1`.

In [29]:
%%writefile app.py

import streamlit as st
from pypdf import PdfReader
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg
import os

st.set_page_config(page_title="Resume/Job Matcher")
st.title("Resume / Job-Matching Agent")

os.environ["GROQ_API_KEY"] = st.secrets["GROQ_API_KEY"]

llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.3)

class ResumeParserTool(BaseTool):
    name: str = "Resume Parser"
    description: str = "Extracts skills, experience, and education from resume text or PDF."
    def _run(self, resume_text: str) -> str:
        parser_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.1)
        prompt = f"""Extract Skills, Experience, and Education as clean bullet points:\n\n{resume_text}"""
        return parser_llm.call(prompt)

class JobMatchComparatorTool(BaseTool):
    name: str = "Job Match Comparator"
    description: str = "Compares parsed resume data against a job description and returns a match score with gaps."
    def _run(self, parsed_resume: str, job_description: str) -> str:
        comparator_llm = LLM(model="groq/openai/gpt-oss-120b", temperature=0.2)
        prompt = f"""Compare resume vs job description. Return match score /100, matches, gaps, recommendation.
        Parsed Resume:\n{parsed_resume}\n\nJob Description:\n{job_description}"""
        return comparator_llm.call(prompt)

resume_text = ""
input_method = st.radio("How will you provide the resume?", ["Paste Text", "Upload PDF"])

if input_method == "Paste Text":
    resume_text = st.text_area("Paste resume here", height=200)
else:
    uploaded_file = st.file_uploader("Upload resume PDF", type=["pdf"])
    if uploaded_file:
        reader = PdfReader(uploaded_file)
        resume_text = "\n".join([page.extract_text() or "" for page in reader.pages])

job_description = st.text_area("Paste job description here", height=150)

if st.button("Match") and resume_text and job_description:
    with st.spinner("Agent is working..."):
        matcher_agent = Agent(
            role="Resume Matching Specialist",
            goal="Parse resumes and accurately match them against job descriptions",
            backstory="An experienced HR tech specialist who evaluates resumes against job requirements with precision.",
            tools=[ResumeParserTool(), JobMatchComparatorTool()],
            llm=llm,
            verbose=True
        )
        matching_task = Task(
            description=(
                "Given the resume text: {resume_text} and job description: {job_description}, "
                "first parse the resume using the Resume Parser tool, "
                "then compare the parsed result against the job description using the Job Match Comparator tool. "
                "Return the final match report."
            ),
            expected_output="A match score, matching points, missing points, and a recommendation.",
            agent=matcher_agent
        )
        crew = Crew(agents=[matcher_agent], tasks=[matching_task], verbose=True)
        result = crew.kickoff(inputs={"resume_text": resume_text, "job_description": job_description})

    st.markdown("### Match Report")
    st.markdown(str(result))

Overwriting app.py


This cell uses the `%%writefile` magic command to create an `app.py` file. This file contains the complete Streamlit application code, including the UI for inputting resumes (text or PDF) and job descriptions, and integrating the CrewAI agents for matching. The output `Overwriting app.py` indicates the file was successfully written or updated.

In [30]:
!head -5 app.py


import streamlit as st
from pypdf import PdfReader
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool


This cell displays the first 5 lines of the `app.py` file, allowing a quick verification of its content.

In [31]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{userdata.get("GROQ_API_KEY")}"')

This cell creates a `.streamlit` directory if it doesn't exist and then writes the `GROQ_API_KEY` (retrieved from Colab secrets) into a `secrets.toml` file within that directory. This makes the API key securely accessible to the Streamlit application.

In [32]:
!pkill -f streamlit

This cell executes a shell command to gracefully stop any running Streamlit processes. This is useful for ensuring a clean restart of the Streamlit application.

In [33]:
!streamlit run app.py --server.port 8501 &> streamlit_log.txt &
import time
time.sleep(10)
!cat streamlit_log.txt



2026-08-06 08:27:52.252 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.252.127.18:8501



This cell starts the Streamlit application (`app.py`) on port 8501 in the background (`&`). It redirects output to `streamlit_log.txt`. After a 10-second delay, it prints the content of `streamlit_log.txt`, showing the Streamlit server has started. The `ngrok` warnings about connection refused are expected as Streamlit is just starting and `ngrok` might try to connect before the server is fully up.

In [34]:
!pip install pyngrok -q

This cell quietly installs the `pyngrok` library, which is used to create secure public URLs for applications running locally.

In [35]:
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
public_url = ngrok.connect(8501)
print("Your app is live :", public_url)

Your app is live : NgrokTunnel: "https://postnasal-angled-sessions.ngrok-free.dev" -> "http://localhost:8501"


This cell imports `ngrok`, sets the authentication token using a Colab secret, and then establishes a public URL tunnel to the Streamlit app running on port 8501. Finally, it prints the live public URL for the application.